# Lab 1 - PyTerrier introduction

In this notebook, we provide an introduction to [PyTerrier](https://github.com/terrier-org/pyterrier). This lab is an unassessed exercise, as preparation for the main exercises.

The intended learning outcome of this lab is to provide an introduction to PyTerrier as well as an illustration of some fundamental concepts in IR (this will be followed by a PyTerrier tutorial in the class).
  - installation
  - indexing a collection
  - accessing an index
  - the `terrier.Retriever` transformer for searching an index
  - reformulating queries
  - composing transformers together to make retrieval pipelines

Pre-requisites:
 - We assume that you are confident in programming Python, including [lambda functions](https://realpython.com/python-lambda/).
 - We assume that you have completed the pre-sessional notebook, which includes a Pandas refresher.
 - We will **only be supporting notebooks on the Google Colab platform**.
 - You can also try to use your own personal machine, but if you have problems, you should resort to using Colab.

Related Reading:
 - [Pandas documentation](https://pandas.pydata.org/docs/)
 - [PyTerrier documentation](https://pyterrier.readthedocs.io/en/latest/)

Efforts Required:
 - We don't expect everyone to complete this lab within their allotted lab hour. You will need to spend some time on it outside of the lab. Exercise 1 (assessed), builds on the learning outcomes of this lab, so it's essential you complete this Lab before starting on Exercise 1. We recommend that you complete this Lab before 5 February 2026.



In the following, we aim to illustrate concepts we will be studying in the early lectures of the course.

PyTerrier is a Python framework, but uses the underlying [Terrier.org information retrieval toolkit](http://terrier.org) for many indexing and retrieval operations. While PyTerrier is a recent framework (launched in 2020), Terrier is written in Java and has a long history dating back to 2001. PyTerrier makes it easy to perform modern IR experiments in Python, but uses the mature Terrier platform for the expensive indexing and retrieval operations.

In the following, we introduce everything you need to know about PyTerrier (which we will build on in the PyTerrier tutorial and both Ex1 & Ex2), and also provide appropriate links to relevant parts of the [PyTerrier documentation](https://pyterrier.readthedocs.io/en/latest/).


### Installation & Configuration

PyTerrier is installed as follows. This might take a few minutes, so you can read on.

In [ ]:
%pip install -q git+https://github.com/cmacdonald/irhm.git
import pyterrier as pt

### Documents, Indexing and Indexes

Much of PyTerrier's view of the world is wrapped up in Pandas dataframes. Let's consider some textual documents in a dataframe.


In [ ]:
import pandas as pd
docs_df = pd.DataFrame([
    {"docno": "d1", "text": "this is the first document of many documents"},
    {"docno": "d2", "text": "this is another document"},
    {"docno": "d3", "text": "the topic of this document is unknown"}
])
docs_df

Before any search engine can estimate which documents are most likely to be relevant for a given query, it must index the documents. The index is written into a directory called `index_3docs`.

In [ ]:
index = pt.terrier.TerrierIndex("index_3docs")
indexer = index.indexer()
indexer.index(docs_df.to_dict(orient="records"))

The documents are now indexed in the `index_3docs` directory. We can look in it and see that it has created several small files. The filenames give hints to the corresponding data structures:
 - some files relating to a lexicon (aka vocabulary)
 - a file relating to the inverted index
 - a file relating to the direct index (aka forward index, used for QE)
 - files relating to document metadata (e.g. storing the unique identifiers for documents) and document lengths


In [ ]:
!ls -lh index_3docs/

We also have a reference to it as a Python `TerrierIndex` object.

In the documentation for [`TerrierIndex`](https://pyterrier.readthedocs.io/en/latest/terrier/api.html#pyterrier.terrier.TerrierIndex), you'll find a bunch of useful methods. Some let you build transformers for the index --- we'll get to those later. Others let you read the data from the index, e.g.:
 - [`index.collection_statistics()`](https://pyterrier.readthedocs.io/en/latest/terrier/api.html#pyterrier.terrier.TerrierIndex.collection_statistics)
 - [`index.inverted_index()`](https://pyterrier.readthedocs.io/en/latest/terrier/api.html#pyterrier.terrier.TerrierIndex.inverted_index)
 - [`index.lexicon()`](https://pyterrier.readthedocs.io/en/latest/terrier/api.html#pyterrier.terrier.TerrierIndex.lexicon)

Let's see what's in the collection statistics:

In [ ]:
print(index.collection_statistics())

Ok, that seems fair - we have 3 documents. But why only 4 terms?
Let's check the [Lexicon](http://terrier.org/docs/current/javadoc/org/terrier/structures/Lexicon.html), which is our vocabulary. Fortunately, the Lexicon can be iterated easily from Python:

In [ ]:
for term, lex_entry in index.lexicon():
  print(f"{term} ({type(term)}) -> {lex_entry} ({type(lex_entry)})")

Here, iterating over the Lexicon returns a tuple of a String term, and a [`LexiconEntry`](http://terrier.org/docs/current/javadoc/org/terrier/structures/LexiconEntry.html) -- which itself is an [EntryStatistics](http://terrier.org/docs/current/javadoc/org/terrier/structures/EntryStatistics.html) -- and contains information including the statistics of that term.

> Note that `LexiconEntry`, along with some other values returned from the index, are Python wrappers around Java objects. You can tell based on the class name: `<class 'jnius.reflect.[JAVA_CLASS_NAME]'>`. You may need to check the [Terrier Java documentation](http://terrier.org/docs/current/javadoc) for information about how to use those.


So what did we find? Here are some observations:
 - we only have 4 unique terms, as stopwords were removed
 - we have one term for `"document"`, even though `"documents"` occurred in document `"d1"`.

Both these observations make sense. Indeed, by default, Terrier removes standard stopwords and applies Porter's stemmer.

Further:
 - `Nt` is the number of unique documents that each term occurs in. This is useful for calculating IDF (c.f. Lecture 3)
 - `TF` is the total number of occurrences - some weighting models use this instead of Nt.
 - The numbers in the `@{}` are a pointer - they tell Terrier where the postings are for that term in the inverted index data structure.

Finally, we can also use the square bracket notation to lookup terms in Terrier's lexicon. This is equivalent to [Lexicon.getLexiconEntry("chemical")](http://terrier.org/docs/current/javadoc/org/terrier/structures/Lexicon.html#getLexiconEntry(KEY))


In [ ]:
print(index.lexicon()["document"])

Let's now think about the inverted index. Remember from Lecture 2 that the inverted index tells us in which *documents* each term occurs in. The `LexiconEntry` is the pointer that tells us where to find the postings for that term in the inverted index.

In [ ]:
pointer = index.lexicon()["document"]
for posting in index.inverted_index().getPostings(pointer):
  print(f"{posting} doclen={posting.getDocumentLength()}")

Ok, so we can see that `"document"` occurs once in each of the three documents. NB: Terrier counts documents as integers from 0 (called *docids*). It records the mapping back to *docnos* (the string form, i.e. `"d1"`, `"d2"`) in a separate data structure called the MetaIndex.

Similarly, we can see what terms occur in document d1, by using a datastructure called the *direct* index. You can think of this as the opposite of the inverted index: the inverted index tells us, for each term, which documents contain that term; the direct index tells us, for each document, which terms it contains.

Let's define a function to show us the contents of a document using the direct index. This a translation of [the Java code for displaying the contents of a document in the Terrier documentation](https://github.com/terrier-org/terrier-core/blob/5.x/doc/extend_retrieval.md#what-terms-occur-in-the-11th-document). When using our function, we can specify which document either by docno (external string) or docid (internal int).

In [ ]:
def get_document_direct(index, docno=None, docid=None):
  if docid is None and docno is None:
    raise ValueError("Must specify docno or docid")
  if docno is not None:
    docid = index.meta_index().getDocument("docno", docno)
  else:
    docno = index.meta_index().getItem("docno", docid)
  rtr = f"Docno {docno} (docid {docid})\n"
  pointer = index.document_index().getDocumentEntry(docid)
  for p in index.direct_index().getPostings(pointer):
    termid = p.getId()
    term = index.lexicon()[termid].getKey()
    rtr += f"\t{term} {p.getFrequency()}\n"
  return rtr

In [ ]:
print(get_document_direct(index, docno="d1"))

In [ ]:
print(get_document_direct(index, docid=0))

### Searching an Index

Searching over Terrier indexes is performed using [`pt.terrier.Retriever`](https://pyterrier.readthedocs.io/en/latest/terrier/api.html#pyterrier.terrier.Retriever). It can be constructed using convenience methods on `TerrierIndex`, such as [`index.retriever()`](https://pyterrier.readthedocs.io/en/latest/terrier/api.html#pyterrier.terrier.TerrierIndex.retriever), [`index.tf()`](https://pyterrier.readthedocs.io/en/latest/terrier/api.html#pyterrier.terrier.TerrierIndex.tf), [`index.bm25()`](https://pyterrier.readthedocs.io/en/latest/terrier/api.html#pyterrier.terrier.TerrierIndex.bm25), etc.

In this example, we construct a retriever using the Term Frequency (`tf`) weighting model, then search for a single-word query: `"document"`.

In [ ]:
retriever = index.tf()
retriever.search("document")

So `search()` returns a dataframe with columns:
 - qid: this is by default "1", since it's our first and only query
 - docid: Terrier' internal integer for each doucment
 - docno: the external (string) unique identifier for each document
 - score: the total frequency of the query (terms) in each document
 - rank: A handy attribute showing the descending order by score
 - query: the input query

As expected, the Tf weighting model used here only counts the frequencies of the query terms in each document, i.e.:
$$
score(d,Q) = \sum_{t \in Q} tf_{t,d}
$$

Hence, it's clear that document d1 should be the highest scored document with two occurrences (c.f. `'document'` and `'documents'`) since we are using stemming.

We can also pass a dataframe of one or more queries to the `transform()` method (rather than the `search()` method) of a transformer, with queries numbered "q1", "q2" etc..

In [ ]:
queries = pd.DataFrame([
    {"qid": "q1", "query": "document"},
    {"qid": "q2", "query": "first document"}
])
retriever.transform(queries)

In fact, we are usually calling the `transform()` method, so it's the default method - i.e.
`retriever.transform(queries)` can be more succinctly written as `retriever(queries)`.

In [ ]:
retriever(queries)

### Moving on up

OK, having 3 documents is quite trivial, so let's move up to a slightly larger corpus of documents. We'll be using a corpus of documents called Vaswani NPL. Terrier has a handy [Artifacts API](https://pyterrier.readthedocs.io/en/latest/artifacts/index.html), which allows us to download a ready-made index from sources like HuggingFace.

In [ ]:
index = pt.Artifact.from_hf("pyterrier/vaswani.terrier")

#### **Task 1**
- What are the statistics of our index? You can print the index collection statistics for Vaswani NPL.
- Can you get a feeling for what kind of documents this index contains by inspecting a few documents. e.g. look into the content of the index of docids 0, 10 and 100

In [ ]:
# Task 1: Index statistics and document inspection

print("="*60)
print("VASWANI INDEX COLLECTION STATISTICS")
print("="*60)
print(index.collection_statistics())
print("="*60)

print("\n" + "="*60)
print("INSPECTING DOCUMENTS")
print("="*60)

# Inspect documents at docids 0, 10, and 100
for docid in [0, 10, 100]:
    print(f"\n{get_document_direct(index, docid=docid)}")
    print("-"*60)

So Vaswani has a reasonable index - let's examine the extent that it exhibits some of the textual properties we would expect.

#### **Task 2**

In this task, we will be examining the extent to which the Vaswani NPL collection exhibits some of the textual properties we would expect (see Lecture 3).

Use a [matplotlib](https://matplotlib.org/) histogram to examine the distribution of the terms occurring in the Lexicon - i.e. it should be possible to read for a given frequency range, the number of terms that have that frequency.

Hint: What method of the LexiconEntry can be used to determine the frequency for each term?

Hint: it might be easier to read if you apply a logarithmic scale for the y-axis. Allow matplotlib to use its default number of bins (which is 10).

<p>&nbsp;</p>

#### **Stretch Task (Optional, do this LAST)**

Do this once you have completed the other tasks...

Next, let's try to see if Vaswani follows a Zipfian distribution. In terms of probabilities, we have
$$
r.Pr_t \approx A
$$
where $r$ is the rank of the term, and $Pr_t$ is the probability of the term. To check how well Vaswani follows Zipf, reproduce slide 14 from Lecture 3 (concerning AP89) for this collection. We expect that Zipf's law will not hold for low and high ranked terms.

Hint: SciPy's [rankdata()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rankdata.html) function might be useful for getting the rank of any term frequency.

Hint: To obtain $Pr_t$, you should compare each term's frequency to one of the collection statistics.


In [ ]:
# Task 2: Histogram of term distribution

import matplotlib.pyplot as plt
import numpy as np

print("="*60)
print("TASK 2: TERM FREQUENCY DISTRIBUTION")
print("="*60)

# Collect term frequencies from the lexicon
term_frequencies = []
for term, lex_entry in index.lexicon():
    # Use getFrequency() to get the total frequency of the term across all documents
    term_frequencies.append(lex_entry.getFrequency())

print(f"Total unique terms in lexicon: {len(term_frequencies)}")
print(f"Min frequency: {min(term_frequencies)}")
print(f"Max frequency: {max(term_frequencies)}")
print(f"Mean frequency: {np.mean(term_frequencies):.2f}")

# Create histogram with logarithmic y-axis
plt.figure(figsize=(10, 6))
plt.hist(term_frequencies, bins=10, edgecolor='black', alpha=0.7)
plt.yscale('log')  # Logarithmic scale for y-axis
plt.xlabel('Term Frequency', fontsize=12)
plt.ylabel('Number of Terms (log scale)', fontsize=12)
plt.title('Distribution of Term Frequencies in Vaswani Collection', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservation: The distribution shows that most terms have low frequencies,")
print("while a few terms have very high frequencies - a typical characteristic of text collections.")

In [ ]:
# Task 2 STRETCH: Zipf's Law Analysis

from scipy.stats import rankdata

print("="*60)
print("TASK 2 STRETCH: ZIPF'S LAW ANALYSIS")
print("="*60)

# Get term frequencies and calculate probabilities
term_data = []

# Get total tokens from collection statistics (it's a Java object, not a dict)
coll_stats = index.collection_statistics()
total_tokens = coll_stats.getNumberOfTokens()

for term, lex_entry in index.lexicon():
    freq = lex_entry.getFrequency()
    prob = freq / total_tokens  # Pr_t = frequency / total tokens
    term_data.append({'term': term, 'freq': freq, 'prob': prob})

# Sort by frequency (descending) to get ranks
term_data_sorted = sorted(term_data, key=lambda x: x['freq'], reverse=True)

# Assign ranks
ranks = list(range(1, len(term_data_sorted) + 1))
probs = [t['prob'] for t in term_data_sorted]

# Calculate r * Pr_t
r_times_pr = [r * p for r, p in zip(ranks, probs)]

# Plot Zipf's law: r * Pr_t vs rank
plt.figure(figsize=(12, 6))
plt.plot(ranks, r_times_pr, 'b.', markersize=2, alpha=0.5)
plt.xlabel('Rank (r)', fontsize=12)
plt.ylabel('r × Pr(term)', fontsize=12)
plt.title("Zipf's Law: r × Pr(term) vs Rank for Vaswani Collection", fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=np.median(r_times_pr), color='r', linestyle='--', 
            label=f'Median = {np.median(r_times_pr):.6f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nTotal tokens: {total_tokens:,}")
print(f"Unique terms: {len(term_data_sorted):,}")
print(f"Median r×Pr_t: {np.median(r_times_pr):.6f}")
print(f"Mean r×Pr_t: {np.mean(r_times_pr):.6f}")
print("\nObservation: If Zipf's law held perfectly, r×Pr_t would be constant (horizontal line).")
print("We expect deviation for low-ranked (very common) and high-ranked (very rare) terms.")

Next, Vaswani is a *test collection*, in that it is not just a corpus of documents, but it also has a set of queries (called *topics*), with each query having one or more **known** *relevant* documents (also called *qrels*). We'll hear more about test collections later in Lecture 5.

We will first load the Vaswani dataset using PyTerrier's [`pt.get_dataset()` API](https://pyterrier.readthedocs.io/en/latest/datasets.html).

We can easily access the topics and qrels from the dataset. Indeed these are expressed as dataframes as well (we use Pandas's [head()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.head.html) method to show only the first 5 topics):

In [ ]:
vaswani_dataset = pt.get_dataset('vaswani')
vaswani_dataset.get_topics().head(5)

In [ ]:
vaswani_dataset.get_qrels().head(5)

#### **Task 3**

In this task you will familiarise yourself with concepts such as topics and qrels, which are key in conducting IR experiments.

Using `get_document_direct()` and the qrels, print the contents of the **relevant** documents for the query with qid "4". NB: qids are strings.

Hint: Try a Pandas apply()?


In [ ]:
# Task 3: Print relevant documents for query "4"

print("="*60)
print('TASK 3: RELEVANT DOCUMENTS FOR QUERY "4"')
print("="*60)

# Get qrels for query "4"
qrels_q4 = vaswani_dataset.get_qrels()
qrels_q4 = qrels_q4[qrels_q4['qid'] == '4']

print(f"\nNumber of relevant documents for query '4': {len(qrels_q4)}\n")

# Print contents of each relevant document using apply
def print_doc_content(row):
    content = get_document_direct(index, docno=row['docno'])
    print(content)
    print("-" * 60)

qrels_q4.apply(print_doc_content, axis=1)

print("\nThese are all the documents marked as relevant (label=1) for query 4.")

### Weighting Models

So far, we have been using the simple Term Frequency as our ranking function for document retrieval. However, we can use other models, such as `"TF_IDF"`, by using the corresponding method on the index.


In [ ]:
tfidf = index.tf_idf()

tfidf(vaswani_dataset.get_topics().head(1))

You will note that, as expected, the scores of documents ranked by TF_IDF are no longer integers. You can see the exact formula used by Terrier from [the Github repo](https://github.com/terrier-org/terrier-core/blob/5.x/modules/core/src/main/java/org/terrier/matching/models/TF_IDF.java#L79).

Terrier supports many weighting models - the documentation contains [a list of supported models](http://terrier.org/docs/current/javadoc/org/terrier/matching/models/package-summary.html) - some of which we will discover later in the IR course.


### PyTerrier Transformers

You'll have noted that terrier.Retriever has a `transform()` method that takes as input a dataframe, and returns another dataframe, which is somehow a *transformation* of the earlier dataframe (e.g. a retrieval transformation). In fact, terrier.Retriever is just one of many similar objects in PyTerrier, which we call [transformers](https://pyterrier.readthedocs.io/en/latest/transformer.html) (represented by the Transformer base class).

Let's give a look at a terrier.Retriever transformer, e.g. the `tfidf` one:

In [ ]:
#tfidf is a transformer...
print(isinstance(tfidf, pt.Transformer))
#Yes!

# this prints the type hierarchy of the TF_IDF class
tfidf.__class__.__mro__

Transformers have some useful methods:
 - they always implement a `.transform(DataFrame) -> DataFrame` method
 - they have a `.search(str) -> DataFrame` method, which allows to execute a single query. A rough implementation of `.search()` is shown below.
    ```python
    def search(self, query : str) -> DataFrame:
        return self.transform(pd.DataFrame([['1', query]], columns=['qid', 'query'])
    ```



Going further, as `tfidf` is a retriever, we know that it expects queries (lets call this a `Q` dataframe, and returns ranked results (an `R` dataframe). Actually, `tfidf` can tell us this if we render it in the notebook - ie put it in a cell by itself.

In [ ]:
tfidf

Hint: hold your mouse over the Q or R dataframes to see the columns, or over TF_IDF block to see the settings of the Retriever.

You can render any transformer as a schematic to better understand it. Here is the TF `retriever` from above. Can you spot the two differences between the Tf and TF_IDF transformer settings?

In [ ]:
retriever

## Operators on PyTerrier Transformers

One interesting capability of all transformers is that they can be combined using Python operators (this is called operator overloading).

Concretely, imagine that you want to chain transformers together - e.g. rank documents first by Tf then re-rank the *exact same* documents by TF.IDF. We can do this using the `>>` operator - we call this composition, or "then". The outcome of the operator is a new transformer.

In [ ]:
# this is our first retrieval transformer -
# it transform a queries dataframe to a results dataframe
tf = index.tf()

tf(vaswani_dataset.get_topics().head(1))

In [ ]:
# now let's define a pipeline
pipeline = tf >> tfidf
print(isinstance(tfidf, pt.Transformer))

pipeline(vaswani_dataset.get_topics().head(1))


Using this pipeline, we are working in a **declarative** manner:
 - We are *declaring* that we first want to apply the `tf` transformer.
 - We then want the `tfidf` transformer to work on the output of the `tf` transformer.

If we render the pipeline in our notebook, this can be seen.

In [ ]:
pipeline

 The whole pipeline is also an instance of Transformer, so has the usual `.transform()`, `.search()` methods etc.:

In [ ]:
pipeline.search("chemical reactions")

There are a number of other PyTerrier operators:


| Operator | Meaning |
|----------|---------|
| `>>`     | Chaining together transformers, aka "Then" / composition|
| `+`	   |Linear combination of scores|
| `*`     |Scalar factoring of scores|
| `&` | Document Set Intersection |
| <code>&#124;</code> |Document Set Union |
| `%` |Apply rank cutoff |
| `^` | Concatenate the output of one transformer with another |
| <code>**</code>  | Feature Union |

Each of these operate on the dataframes returned by one or more transformers, and therefore has clear relational algebra semantics.  We provide worked examples of the input and output of each of the operators in the [PyTerrier documentation on operators](https://pyterrier.readthedocs.io/en/latest/operators.html). **Do take the time to browse this documenation - you will need to use operators in Task 4**.

Operators are a key concept of PyTerrier, and you will need to use operators to solve Task 4.

#### **Task 4**

In this task you will consider the use of the terrier.Retriever transformer for searching an index, and how to combine transformers together to make retrieval
pipelines. In particular, you will be experimenting with the PyTerrier *operators* when manipulating search results.

To improve your understanding of the PyTerrier operators, the task has two parts, asking you to deploy two specific rankers using the PyTerrier operators.


**(a)**

Create a ranker that returns the top 10 scored documents based on the following retrieval model:
$$
score(d,Q) = \big( \sum_{t \in Q} w_{TF\_IDF}(t,d) \big)  + 3 * \sum_{t \in Q} tf_{t,d}
$$
In essence, this equation scores documents by linearly combining the scores of two different retrieval models.

For the correct solution, your pipeline should return no more than 10 documents and document docno `"10703"` should have a score of 10.381090 for query `"chemical"`.

Hint: You do *not* need to access the inverted index directly.

**(b)**

Create a ranker that performs the following:

 - obtains the top 10 highest scoring documents by term frequency (`index.tf()`)
 - obtains the top 10 highest scoring documents by TF.IDF (`index.tf_idf()`)
 - reranks only those documents found in BOTH of the previous retrieval settings using the BM25 weighting model (Indeed BM25 is another famous weighting model that you will hear more about in Lecture 8).

How many documents are retrieved by this full pipeline for the query "chemical".

If you obtain the correct solution, the document with docno "2558" should have a score of 10.322110 for query "chemical".

Hints:
 - each correct solution needs only a few lines of code if you properly use the PyTerrier operators
 - you should not need to perform any Pandas dataframe operations
 - You will need to instantate a BM25 terrier.Retriever

In [ ]:
# Task 4: PyTerrier Operators

print("="*60)
print("TASK 4(a): TF_IDF + 3*TF RANKER (TOP 10)")
print("="*60)

# Create the ranker: TF_IDF + 3*TF, then limit to top 10
# score(d,Q) = TF_IDF(d,Q) + 3 * TF(d,Q)

tf_retriever = index.tf()
tfidf_retriever = index.tf_idf()

# Use operators: + for linear combination, * for scalar multiplication, % for rank cutoff
ranker_4a = (tfidf_retriever + 3 * tf_retriever) % 10

# Test with query "chemical"
result_4a = ranker_4a.search("chemical")
print(result_4a)

# Verify the solution
doc_10703 = result_4a[result_4a['docno'] == '10703']
if len(doc_10703) > 0:
    print(f"\n✓ Verification: Document '10703' score = {doc_10703.iloc[0]['score']:.6f}")
    print(f"  Expected: 10.381090")
else:
    print("\n✗ Document '10703' not found in top 10")

print(f"\n✓ Number of documents returned: {len(result_4a)}")

print("\n" + "="*60)
print("TASK 4(b): (TF ∩ TF_IDF) → BM25 RERANKING")
print("="*60)

# Create ranker that:
# 1. Gets top 10 from TF
# 2. Gets top 10 from TF_IDF
# 3. Intersects the two sets
# 4. Reranks with BM25

bm25_retriever = index.bm25()

# Use operators: % for top-k cutoff, & for intersection, >> for then/composition
ranker_4b = ((tf_retriever % 10) & (tfidf_retriever % 10)) >> bm25_retriever

# Test with query "chemical"
result_4b = ranker_4b.search("chemical")
print(result_4b)

print(f"\n✓ Number of documents returned: {len(result_4b)}")
print(f"  (This is the number of documents in BOTH TF top-10 AND TF_IDF top-10)")

# Verify the solution
doc_2558 = result_4b[result_4b['docno'] == '2558']
if len(doc_2558) > 0:
    print(f"\n✓ Verification: Document '2558' score = {doc_2558.iloc[0]['score']:.6f}")
    print(f"  Expected: 10.322110")
else:
    print("\n✗ Document '2558' not found in results")

print("="*60)

### What is Success?

So far, we have been creating search engine models, but we haven't decided if any of them is actually any good. While we will study *evaluation measures* in Lecture 5, let's just investigate if we are getting a correct ("relevant") document at the first rank.

In [ ]:
qrels = vaswani_dataset.get_qrels()
def get_res_with_labels(ranker, df):
  #get the results for the query or queries
  results = ranker( df )
  # left outer join with the qrels
  with_labels = results.merge(qrels, on=["qid", "docno"], how="left").fillna(0)
  return with_labels

#lets get the Tf results for the first query
get_res_with_labels(tf, vaswani_dataset.get_topics().head(1) )



So, we got a relevant document third ranked using Tf. Let's check TF_IDF to see whether it does better:

In [ ]:
#lets get the tfidf results for the first query
get_res_with_labels(tfidf, vaswani_dataset.get_topics().head(1) )

Interesting - it got a relevant document at the top rank. Think about what TF_IDF is doing compared to Tf.

Keeping with the Tf ranking function, let's try to help it perform better by better formulating the query `'measurement of dielectric constant of liquids by the use of microwave techniques'`.

In particular, Terrier can take into account how often a query term occurs in a query. In doing so, documents are actually scored like this:
$$
score(d,Q) = \sum_{t \in Q} tf_{t,q} * tf_{t,d}
$$
where $tf_{t,d}$ is the frequency of the term in *document* and $tf_{t,q}$ is the frequency of the term in the *query*.

However, in our example query ( `'measurement of dielectric constant of liquids by the use of microwave techniques'`) all (non-stopword) query terms have a query term frequency of 1.

We can try to give more (or less) weight to some of these query terms, so as to place more or less emphasis on those terms, by appending a number such as `^2` (or `^0.1`) to some words in the query string. For example ^2 indicates that the weight of the corresponding term is 2.

In [ ]:
# here I try to change the emphasis on some words
newQueryDf = pd.DataFrame([
  ["1", "measurement^0.1 of dielectric constant of liquids by "+
    "the use of microwave techniques^2"]
  ], columns=["qid", "query"])
get_res_with_labels(tf, newQueryDf )


#### **Task 5**

In this task, you will be familiarising yourself with an important aspect of users’ interaction with IR systems, namely how to reformulate queries in order to improve the effectiveness of the
system.

Spend **up to 5 minutes** trying to reformulate the above query by increasing or decreasing the weight of some query terms to get a relevant result at the first rank.

Hint: You might want to inspect some of the relevant (or non-relevant!) documents to see what words they contain. E.g. you might want to look at the contents of docnos 7923 and 3693.

In [ ]:
# Task 5: Manual Query Reformulation

print("="*60)
print("TASK 5: MANUAL QUERY REFORMULATION")
print("="*60)

# Original query
original_query = "measurement of dielectric constant of liquids by the use of microwave techniques"

print(f"Original query: {original_query}\n")
print("Original query results:")
original_df = pd.DataFrame([["1", original_query]], columns=["qid", "query"])
print(get_res_with_labels(tf, original_df).head(10))

# Let's inspect some relevant and non-relevant documents
print("\n" + "="*60)
print("INSPECTING KEY DOCUMENTS")
print("="*60)

print("\nRelevant document 7923:")
print(get_document_direct(index, docno="7923"))

print("\nRelevant document 3693:")
print(get_document_direct(index, docno="3693"))

# Strategy: Based on inspection, increase weight on technical terms
# and decrease weight on common words

print("\n" + "="*60)
print("REFORMULATED QUERY ATTEMPTS")
print("="*60)

# Attempt 1: Boost "dielectric" and "microwave" as they seem key technical terms
reformulated_query_1 = "measurement^0.5 dielectric^3 constant^2 liquids^0.5 microwave^3 techniques^2"
print(f"\nAttempt 1: {reformulated_query_1}")
reformulated_df_1 = pd.DataFrame([["1", reformulated_query_1]], columns=["qid", "query"])
result_1 = get_res_with_labels(tf, reformulated_df_1).head(5)
print(result_1)

if result_1.iloc[0]['label'] == 1:
    print("✓ SUCCESS! Got a relevant document at rank 1!")
else:
    # Attempt 2: Different weighting
    reformulated_query_2 = "measurement^0.1 dielectric^5 constant^3 liquids microwave^5 techniques^2"
    print(f"\nAttempt 2: {reformulated_query_2}")
    reformulated_df_2 = pd.DataFrame([["1", reformulated_query_2]], columns=["qid", "query"])
    result_2 = get_res_with_labels(tf, reformulated_df_2).head(5)
    print(result_2)
    
    if result_2.iloc[0]['label'] == 1:
        print("✓ SUCCESS! Got a relevant document at rank 1!")
    else:
        # Attempt 3: Focus even more on "microwave" and "dielectric"
        reformulated_query_3 = "dielectric^10 microwave^10 constant^2 liquids techniques"
        print(f"\nAttempt 3: {reformulated_query_3}")
        reformulated_df_3 = pd.DataFrame([["1", reformulated_query_3]], columns=["qid", "query"])
        result_3 = get_res_with_labels(tf, reformulated_df_3).head(5)
        print(result_3)
        
        if result_3.iloc[0]['label'] == 1:
            print("✓ SUCCESS! Got a relevant document at rank 1!")

print("\n" + "="*60)
print("REFLECTION:")
print("Manual query reformulation shows that emphasizing key technical terms")
print("(like 'dielectric' and 'microwave') while de-emphasizing common words")
print("can improve retrieval effectiveness. This insight helps us understand")
print("what automatic query expansion methods should aim to achieve.")
print("="*60)

**Reflection**: While we could not manually reformulate each user's query, however knowing which words/terms should be important or not can help us understand what a good ranking model should do.

Later in the course, we'll talk about other approaches to (automatically) reformulate queries, such as query expansion, which can help to improve the search engine's effectiveness.


## That's all folks


Future IR exercises will build upon the learning outcome from this warmup lab. Should you need to reinforce any of the material in this lab, the following parts of the PyTerrier documentation will be useful:
 * [PyTerrier datasets](https://pyterrier.readthedocs.io/en/latest/datasets.html)
 * [Using Terrier for retrieval](https://pyterrier.readthedocs.io/en/latest/terrier-retrieval.html)
 * [Transformers in PyTerrier](https://pyterrier.readthedocs.io/en/latest/transformer.html)
 * [Transformer Operators](https://pyterrier.readthedocs.io/en/latest/operators.html)